In [1]:
from crewai import Agent, Task, Crew
from crewai.llm import LLM
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import duckdb
from langchain_openai import ChatOpenAI
import os

In [2]:
api_key=os.environ["OPENAI_API_KEY"]

In [3]:
product_links = [
    "https://www.amazon.com/dp/B09G3HRMVB",
    "https://www.ebay.com/itm/334567890123",
    "https://www.aliexpress.com/item/1005005678901234.html",
    "https://www.etsy.com/listing/123456789/handmade-ceramic-mug",
    "https://www.walmart.com/ip/987654321",
    "https://www.target.com/p/-/A-12345678",
    "https://www.bestbuy.com/site/sku/1234567.p",
    "https://www.zara.com/us/en/product-p01234567.html",
    "https://www.nike.com/t/air-force-1-shoes-123456",
    "https://www.ikea.com/us/en/p/product-12345678/"
]

In [4]:
product_dataset_url="https://drive.google.com/uc?export=download&id=1s4T0-L4-LgoUCj0WNU1JVH2-jHXDiWf6"

df=pd.read_csv(product_dataset_url)
df

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0
...,...,...,...,...,...,...,...,...,...
32946,a0b7d5a992ccda646f2d34e418fff5a0,moveis_decoracao,45.0,67.0,2.0,12300.0,40.0,40.0,40.0
32947,bf4538d88321d0fd4412a93c974510e6,construcao_ferramentas_iluminacao,41.0,971.0,1.0,1700.0,16.0,19.0,16.0
32948,9a7c6041fa9592d9d9ef6cfe62a71f8c,cama_mesa_banho,50.0,799.0,1.0,1400.0,27.0,7.0,27.0
32949,83808703fc0706a22e264b9d75f04a2e,informatica_acessorios,60.0,156.0,2.0,700.0,31.0,13.0,20.0


In [5]:
schema = "\n".join(
    [f"{col} ({dtype})" for col, dtype in df.dtypes.items()]
)

print(schema)

product_id (str)
product_category_name (str)
product_name_lenght (float64)
product_description_lenght (float64)
product_photos_qty (float64)
product_weight_g (float64)
product_length_cm (float64)
product_height_cm (float64)
product_width_cm (float64)


In [6]:
#Json database we will workon
#json_df=df.to_json(orient="records") #this is human friendly

In [7]:
question="what are the products names in table1?"

In [8]:
#LLM
llm=LLM(model="gpt-4o-mini")  #new crewAI does not use Langchain directly

In [9]:
#This agent will be responsible for writing SQL queries
sql_agent=Agent(
    role="Senior Data Analyst & SQL Expert",
    
    goal="""
    Translate natural language questions into accurate and efficient SQL queries.
    """,
    
    backstory="""You are an expert data analyst with deep knowledge of SQL.
    You specialize in understanding business questions and converting them into precise SQL queries.
    You always ensure correctness, efficiency, and alignment with the database schema.
   """,
    llm=llm
)

In [10]:
#Now we include Tasks

SQL_task=Task(
    description=f"""
    Given the question:
    {question}
    
    -Translate this question into accurate and efficient SQL query.
    -You are working with a SQL table called "table1".
    -table1 schema:
    {schema}

    RULES:
    - Only use columns from the schema above
    - Do not invent tables or columns
    - Write efficient SQL queries
    - Return ONLY the SQL query
    - Do NOT include markdown formatting
    - Do NOT wrap output in ``` or ```sql
    - Do NOT add explanations
    - Do NOT label the output in any way
    - Output must start with SELECT, INSERT, UPDATE, DELETE, etc.
    - Always ensure the SQL query returns at most 10 rows.
    - You MUST include a LIMIT 10 clause in every query unless the query already guarantees fewer rows (e.g., COUNT, SUM, aggregations).
    - Never return more than 10 rows under any circumstance.
    """,

    expected_output="A valid and executable SQL query based on the question.",
    
    agent=sql_agent   
)



In [11]:
#Create a crew

crew_sql= Crew(
    agents=[sql_agent],
    tasks=[SQL_task],
    verbose=True
)

answer=crew_sql.kickoff().raw
print(answer)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: c3b3dad3-1e84-4810-9861-5602fb42de34                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│      Given the question:                                                                                        │
│      what are the products names in table1?                                                                     │
│                                                                                                                 │
│      -Translate this question into accurate and efficient SQL query.                                            │
│      -You are working with a SQL table called "table1".                                                         │
│      -table1 schema:                                                                                            │
│      product_id (str)                                                                                           │
│  product_category_name (str)                                                                                    │
│  product_name_lenght (float64)                                                                                  │
│  product_description_lenght (float64)                                                                           │
│  product_photos_qty (float64)                                                                                   │
│  product_weight_g (float64)                                                                                     │
│  product_length_cm (float64)                                                                                    │
│  product_height_cm (float64)                                                                                    │
│  product_width_cm (float64)                                                                                     │
│                                                                                                                 │
│      RULES:                                                                                                     │
│      - Only use columns from the schema above                                                                   │
│      - Do not invent tables or columns                                                                          │
│      - Write efficient SQL queries                                                                              │
│      - Return ONLY the SQL query                                                                                │
│      - Do NOT include markdown formatting                                                                       │
│      - Do NOT wrap output in ``` or ```sql                                                                      │
│      - Do NOT add explanations                                                                                  │
│      - Do NOT label the output in any way                                                                       │
│      - Output must start with SELECT, INSERT, UPDATE, DELETE, etc.                                              │
│      - Always ensure the SQL query returns at most 10 rows.                                                     │
│      - You MUST include a LIMIT 10 clause in every query unless the query already guarantees fewer rows (e.g.,  │
│  COUNT, SUM, aggregations).                                                                                     │
│      - Never return more than 10 rows under any circumstance.                                                   │
│                                                        

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Data Analyst & SQL Expert                                                                        │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Given the question:                                                                                        │
│      what are the products names in table1?                                                                     │
│                                                                                                                 │
│      -Translate this question into accurate and efficient SQL query.                                            │
│      -You are working with a SQL table called "table1".                                                         │
│      -table1 schema:                                                                                            │
│      product_id (str)                                                                                           │
│  product_category_name (str)                                                                                    │
│  product_name_lenght (float64)                                                                                  │
│  product_description_lenght (float64)                                                                           │
│  product_photos_qty (float64)                                                                                   │
│  product_weight_g (float64)                                                                                     │
│  product_length_cm (float64)                                                                                    │
│  product_height_cm (float64)                                                                                    │
│  product_width_cm (float64)                                                                                     │
│                                                                                                                 │
│      RULES:                                                                                                     │
│      - Only use columns from the schema above                                                                   │
│      - Do not invent tables or columns                                                                          │
│      - Write efficient SQL queries                                                                              │
│      - Return ONLY the SQL query                                                                                │
│      - Do NOT include markdown formatting                                                                       │
│      - Do NOT wrap output in ``` or ```sql                                                                      │
│      - Do NOT add explanations                                                                                  │
│      - Do NOT label the output in any way                                                                       │
│      - Output must start with SELECT, INSERT, UPDATE, DELETE, etc.                                              │
│      - Always ensure the SQL query returns at most 10 rows.                                                     │
│      - You MUST include a LIMIT 10 clause in every query unless the query already guarantees fewer rows (e.g.,  │
│  COUNT, SUM, aggregations).                                                                                     │
│      - Never return more than 10 rows under any circums

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Data Analyst & SQL Expert                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  SELECT product_category_name FROM table1 LIMIT 10;                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│      Given the question:                                                                                        │
│      what are the products names in table1?                                                                     │
│                                                                                                                 │
│      -Translate this question into accurate and efficient SQL query.                                            │
│      -You are working with a SQL table called "table1".                                                         │
│      -table1 schema:                                                                                            │
│      product_id (str)                                                                                           │
│  product_category_name (str)                                                                                    │
│  product_name_lenght (float64)                                                                                  │
│  product_description_lenght (float64)                                                                           │
│  product_photos_qty (float64)                                                                                   │
│  product_weight_g (float64)                                                                                     │
│  product_length_cm (float64)                                                                                    │
│  product_height_cm (float64)                                                                                    │
│  product_width_cm (float64)                                                                                     │
│                                                                                                                 │
│      RULES:                                                                                                     │
│      - Only use columns from the schema above                                                                   │
│      - Do not invent tables or columns                                                                          │
│      - Write efficient SQL queries                                                                              │
│      - Return ONLY the SQL query                                                                                │
│      - Do NOT include markdown formatting                                                                       │
│      - Do NOT wrap output in ``` or ```sql                                                                      │
│      - Do NOT add explanations                                                                                  │
│      - Do NOT label the output in any way                                                                       │
│      - Output must start with SELECT, INSERT, UPDATE, DELETE, etc.                                              │
│      - Always ensure the SQL query returns at most 10 rows.                                                     │
│      - You MUST include a LIMIT 10 clause in every query unless the query already guarantees fewer rows (e.g.,  │
│  COUNT, SUM, aggregations).                                                                                     │
│      - Never return more than 10 rows under any circumstance.                                                   │
│                                                        

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: c3b3dad3-1e84-4810-9861-5602fb42de34                                                                       │
│  Final Output: SELECT product_category_name FROM table1 LIMIT 10;                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

SELECT product_category_name FROM table1 LIMIT 10;


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [12]:
#converting sql into dataframe

con=duckdb.connect()
data=con.register('table1', df)
result=data.query(answer).df().to_json(orient="records") #this is human friendly
print(result)

[{"product_category_name":"perfumaria"},{"product_category_name":"artes"},{"product_category_name":"esporte_lazer"},{"product_category_name":"bebes"},{"product_category_name":"utilidades_domesticas"},{"product_category_name":"instrumentos_musicais"},{"product_category_name":"cool_stuff"},{"product_category_name":"moveis_decoracao"},{"product_category_name":"eletrodomesticos"},{"product_category_name":"brinquedos"}]


In [13]:
#second agent

translator_agent= Agent(
    role="Professional Python Analyst",
    goal="""Your goal is to translate the given output from Json data into into an understandable human language from a given question""",
    backstory= """You have more than 20 years of experience as a Json data translator""",
    llm=llm
)



translator_task=Task(
    description=f"""
    based on the question: {question}
    
    Translate the given output from Json data:
    {result}
    
    into an understandable human language from a given question""",

    expected_output="text",

    agent=translator_agent
)

In [14]:
#Create a crew

crew_translator= Crew(
    agents=[translator_agent],
    tasks=[translator_task],
    verbose=True
)

answer2=crew_translator.kickoff().raw  #raw ensures output is a string
print(answer2)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 2070e8b1-3b1b-4d59-8eaa-17acd2de4fb3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│      based on the question: what are the products names in table1?                                              │
│                                                                                                                 │
│      Translate the given output from Json data:                                                                 │
│      [{"product_category_name":"perfumaria"},{"product_category_name":"artes"},{"product_category_name":"espor  │
│  te_lazer"},{"product_category_name":"bebes"},{"product_category_name":"utilidades_domesticas"},{"product_cate  │
│  gory_name":"instrumentos_musicais"},{"product_category_name":"cool_stuff"},{"product_category_name":"moveis_d  │
│  ecoracao"},{"product_category_name":"eletrodomesticos"},{"product_category_name":"brinquedos"}]                │
│                                                                                                                 │
│      into an understandable human language from a given question                                                │
│  ID: f2d08bcd-e50c-4ca7-8d44-eeb6fdb7fcb0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Professional Python Analyst                                                                             │
│                                                                                                                 │
│  Task:                                                                                                          │
│      based on the question: what are the products names in table1?                                              │
│                                                                                                                 │
│      Translate the given output from Json data:                                                                 │
│      [{"product_category_name":"perfumaria"},{"product_category_name":"artes"},{"product_category_name":"espor  │
│  te_lazer"},{"product_category_name":"bebes"},{"product_category_name":"utilidades_domesticas"},{"product_cate  │
│  gory_name":"instrumentos_musicais"},{"product_category_name":"cool_stuff"},{"product_category_name":"moveis_d  │
│  ecoracao"},{"product_category_name":"eletrodomesticos"},{"product_category_name":"brinquedos"}]                │
│                                                                                                                 │
│      into an understandable human language from a given question                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Professional Python Analyst                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The products in table1 include the following categories: perfumery, arts, sports and leisure, baby products,   │
│  household utilities, musical instruments, cool stuff, furniture and decoration, home appliances, and toys.     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│      based on the question: what are the products names in table1?                                              │
│                                                                                                                 │
│      Translate the given output from Json data:                                                                 │
│      [{"product_category_name":"perfumaria"},{"product_category_name":"artes"},{"product_category_name":"espor  │
│  te_lazer"},{"product_category_name":"bebes"},{"product_category_name":"utilidades_domesticas"},{"product_cate  │
│  gory_name":"instrumentos_musicais"},{"product_category_name":"cool_stuff"},{"product_category_name":"moveis_d  │
│  ecoracao"},{"product_category_name":"eletrodomesticos"},{"product_category_name":"brinquedos"}]                │
│                                                                                                                 │
│      into an understandable human language from a given question                                                │
│  Agent: Professional Python Analyst                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 2070e8b1-3b1b-4d59-8eaa-17acd2de4fb3                                                                       │
│  Final Output: The products in table1 include the following categories: perfumery, arts, sports and leisure,    │
│  baby products, household utilities, musical instruments, cool stuff, furniture and decoration, home            │
│  appliances, and toys.                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

The products in table1 include the following categories: perfumery, arts, sports and leisure, baby products, household utilities, musical instruments, cool stuff, furniture and decoration, home appliances, and toys.


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [16]:
from langchain_openai import ChatOpenAI
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain.agents import create_sql_agent
from langchain.agents.agent_types import AgentType

# IMPORTANT: now works because duckdb-engine is installed
db = SQLDatabase.from_uri("duckdb:///my_db.duckdb")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

toolkit = SQLDatabaseToolkit(db=db, llm=llm)

agent = create_sql_agent(
    llm=llm,
    toolkit=toolkit,
    agent_type=AgentType.OPENAI_FUNCTIONS,
    verbose=True
)

result = agent.invoke({"input": "What are the top 5 products by revenue?"})

print(result["output"])

ImportError: cannot import name 'ExecutionInfo' from 'langgraph.runtime' (/workspaces/MachineLearning-and-Artificial-Intelligence-Engineering/ai_env/lib/python3.12/site-packages/langgraph/runtime.py)